# Reproduce an OpenVector Bench corpus

A corpus here is not a pile of files: it is a **signed Merkle manifest** plus
interchangeable ways to satisfy its hashes — deterministic regeneration, a
cache, a mirror. This notebook runs that claim end to end (`spec/DISTRIBUTION.md` §6):

1. **publish** a corpus (materialize shards, hash them into a manifest);
2. **delete everything** except the manifest;
3. **reconstruct** from a deliberate mixture of regeneration, cache misses, and fetches;
4. **verify**: Merkle roots, byte identity, identical exact-k-NN answers, tamper rejection.

Run top to bottom with no credentials: the demo uses a `file://` mirror. To point it
at a published corpus instead, see the last section.

## Credentials (all optional, all ambient)

| backend | what you need | where it is read from |
|---|---|---|
| local regeneration / `file://` / `https://` | nothing | — |
| S3-compatible mirror (NRP Ceph, AWS, MinIO) | access key + secret | boto3's standard chain: env `AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY` or `~/.aws`; non-AWS endpoint via env `OVB_S3_ENDPOINT` |
| rebuild tiers on NRP | NRP account (kubeconfig) | your `~/.kube/config` |

**Never paste a secret into a cell** — executed notebooks save their inputs, and
notebooks with pasted keys end up public constantly. Everything below reads
credentials from the environment only.

In [ ]:
import hashlib, json, os, shutil
from pathlib import Path
import numpy as np

from openvector_bench import (
    build_manifest, load_manifest, reconstruct, summarize,
    verify_manifest, verify_shard_file, write_manifest,
)
from openvector_bench.generators import philox_u8
from openvector_bench.manifest import shard_filename

WORK = "ovb_demo"                      # everything below lands here
PARAMS = {"rows": 100_000, "dim": 128, "salt": "ovb-notebook-demo"}
N_SHARDS = 8
IMPL = "refcorpus.philox-u8:v0"

In [ ]:
# 1. Publish: materialize shards, mirror them, hash into a manifest.
orig, mirror = os.path.join(WORK, "original"), os.path.join(WORK, "mirror")
for d in (orig, mirror):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
paths = []
for i in range(N_SHARDS):
    data = philox_u8(i, PARAMS)
    p = os.path.join(orig, f"shard_{i:05d}.bin")
    open(p, "wb").write(data); shutil.copy(p, mirror); paths.append(p)

man = build_manifest(
    paths, corpus="ovb-demo", version="0.1.0", metric="cosine",
    dim=PARAMS["dim"], dtype="uint8", rows_per_shard=PARAMS["rows"],
    n_rows=PARAMS["rows"] * N_SHARDS,
    generator={"impl": IMPL, "params": PARAMS, "seed_scheme": "shard_index"},
    sources=[{"kind": "mirror", "role": "cache",
              "url": Path(mirror).resolve().as_uri()}],
)
write_manifest(man, os.path.join(WORK, "manifest.json"))
original_sha = {i: hashlib.sha256(open(p, "rb").read()).hexdigest()
                for i, p in enumerate(paths)}
print(f"published {N_SHARDS} shards, corpus root {man['root'][:16]}…")
# For a real corpus, also: sign_manifest(path, key) — the same GPG identity
# that signs this repository's commits.

In [ ]:
# 2. The deletion. Only the manifest (and the mirror, playing the role of
# a remote cache) survives.
shutil.rmtree(orig)
man = load_manifest(os.path.join(WORK, "manifest.json"))
verify_manifest(man)   # internal consistency before trusting anything
print("originals deleted; manifest is self-consistent")

In [ ]:
# 3. Reconstruct from a FORCED mixture: odd shards regenerate with a wrong
# salt (simulating toolchain drift), so their hashes miss and they fall
# through to the mirror. A miss is a recorded cache miss, never an error.
def drifted(i, p):
    return philox_u8(i, {**p, "salt": p["salt"] + ("-drift" if i % 2 else "")})

recon = os.path.join(WORK, "recon")
shutil.rmtree(recon, ignore_errors=True)
events = reconstruct(man, recon, generators={IMPL: drifted})
summary = summarize(events)
print(json.dumps(summary, indent=2))

In [ ]:
# 4. Verify: every shard against its Merkle root, byte identity with the
# originals, and tamper rejection.
for s in man["shards"]:
    verify_shard_file(man, s["i"], os.path.join(recon, shard_filename(man, s["i"])))
assert all(
    hashlib.sha256(open(os.path.join(recon, shard_filename(man, i)), "rb").read()).hexdigest() == h
    for i, h in original_sha.items()
), "reconstructed bytes differ from originals"

bad = bytearray(open(os.path.join(recon, shard_filename(man, 0)), "rb").read())
bad[12345] ^= 1
open(os.path.join(WORK, "tampered.bin"), "wb").write(bytes(bad))
try:
    verify_shard_file(man, 0, os.path.join(WORK, "tampered.bin"))
    raise AssertionError("tampering was NOT detected")
except ValueError as e:
    print("tamper detected and rejected:", e)
print("all criteria hold: the corpus survived deletion as a manifest")

## Pointing this at a published corpus

Replace the publish cell with a fetch of the real manifest, verify its
signature, then reconstruct exactly as above:

```python
man = load_manifest("manifest.json")        # from the Zenodo DOI / repo release
verify_signature("manifest.json")           # GPG, against the published key
verify_manifest(man)
events = reconstruct(man, "shards/")        # regenerate where possible, fetch the rest
```

The manifest's `sources` list already carries the fallback order (NRP S3 cache →
durable mirror → origin). `s3://` URLs use your ambient AWS-style credentials;
for NRP Ceph or any non-AWS endpoint set `OVB_S3_ENDPOINT`, e.g.
`https://s3-west.nrp-nautilus.io`.

**Rebuilding large tiers on someone else's compute.** Regeneration is
deterministic, so *where* it runs is irrelevant — the hashes decide. With an NRP
account, template a k8s Job per shard that runs `reconstruct_shard` into your
own PVC or bucket; with an AWS key, the same loop on spot instances into your own
S3 bucket. Both produce artifacts identical to the canonical ones, verifiably.
The harness variant of this notebook is
`harness/distribution/reconstruct_experiment.py` — the registered §6 experiment
with pass criteria and a machine-readable report.